This notebook can be used to verify the dataframes displayed values in Napari. 

In [ ]:
import napari
import pandas as pd
from pathlib import Path
import numpy as np
from skimage import io
from glob import glob
import tifffile
#viewer = napari.Viewer()
pd.set_option('display.max_columns', None)

In [ ]:
paths = glob(fr'F:\Mara BUD analyses\buddy-sander\buddy\EW_cell_cycle_assignment\*.csv')
dfs = []
for path in paths:
    name= Path(path).stem
    if name.find('NA')==0:
        strain, cc_phase = 'multiple', 'NA'
    else:
        strain, cc_phase= str(name[:-14]).split('_')
    df = pd.read_csv(path)
    df['cell_cycle_phase'] = cc_phase
    dfs.append(df)
sf = pd.concat(dfs)

In [ ]:
df = pd.read_csv('Mara_budSegmentationData20260311.csv')
df = df.drop('Unnamed: 0',axis=1)

Construct unique identifier used to select cells for displaying in Napari.

In [ ]:
df['fov']=df['source_folder'].str[-2:]
df['uid'] = (
    df['strain'].astype(str) + '_' +
    df['replicate'].astype(str) + '_' +
    df['image_cell_id'].astype(str) + '_' +
    df['fov'].astype(str)
)

In [ ]:
#showcase only relevant columns
showcase_df = df[['uid','image_cell_id', 'mRNA', 'area',
       'area_bud', 'spots', 'dense_regions', 'decomposed_RNAs',
       'tx', 'nascent_RNAs', 'total_RNAs', 'spots_bud', 'dense_regions_bud',
       'decomposed_RNAs_bud', 'tx_bud', 'nascent_RNAs_bud',
       'total_RNAs_bud', 'mother_nuclei_count','spots_mother', 'total_RNAs_mother',
      'replicate','condition + strain']]

Use uid to select cell.

In [ ]:
uid = df['uid'].values[0]	
rows = df[df['uid']==uid]
basepath = Path(rows['source_folder'].values[0])

# Get bounding box coordinates
y0 = int(rows['bbox-0'].iloc[0])
x0 = int(rows['bbox-1'].iloc[0])
y1 = int(rows['bbox-2'].iloc[0])
x1 = int(rows['bbox-3'].iloc[0])
showcase_df[showcase_df['uid']==uid]

***

#### Cell cycle assignmemt

In [ ]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# Parameters
# --------------------------------------------------
pixel_width = 0.065
pixel_area = pixel_width ** 2

cell_area_limits = [6, 30]
bud_area_limits = [0, 15]
nuc_area_limits = [0, 3.5]
cell_length_limits = [2, 20]
# --------------------------------------------------
# Convert pixel measurements to physical units
# --------------------------------------------------
df["axis_major_length_um"] = (
    df["axis_major_length"] * pixel_width
)

df["area_bud_umsq"] = (
    df["area_bud"] * pixel_area
)
# --------------------------------------------------
# Cells with <= 5 ASH1CLB2 mRNAs
# --------------------------------------------------
uids_wfewCLB2ASH1 = (
    df.loc[
        (df["mRNA"] == "ASH1CLB2") &
        (df["total_RNAs"] <= 5),
        "uid"
    ]
    .tolist()
)
# --------------------------------------------------
# Filter for cells of sensible size: 2–20 µm
# --------------------------------------------------

df_wphase = df.loc[
    (df["axis_major_length_um"] > cell_length_limits[0]) &
    (df["axis_major_length_um"] < cell_length_limits[1])
].copy()
# --------------------------------------------------
# Define cell-cycle phase
# --------------------------------------------------
conditions = [

    # G1
    (
        (df_wphase["area_bud_umsq"] == 0) &
        (df_wphase["mother_nuclei_count"] == 1) &
        (df_wphase["uid"].isin(uids_wfewCLB2ASH1))
    ),

    # S
    (
        (df_wphase["area_bud_umsq"] > 0) &
        (df_wphase["area_bud_umsq"] < 2) &
        (df_wphase["mother_nuclei_count"] == 1) &
        (df_wphase["nuclei_bud_count"] == 0)
    ),

    # G2
    (
        (df_wphase["area_bud_umsq"] >= 2) &
        (df_wphase["area_bud_umsq"] < bud_area_limits[1]) &
        (df_wphase["mother_nuclei_count"] == 1) &
        (df_wphase["nuclei_bud_count"] == 0)
    ),

    # M
    (
        (df_wphase["area_bud_umsq"] > 2) &
        (df_wphase["mother_nuclei_count"] == 1) &
        (df_wphase["nuclei_bud_count"] == 1)
    )
]
# --------------------------------------------------
# Assign phase
# --------------------------------------------------
df_wphase["cell_cycle_phase"] = np.select(
    conditions,
    ["G1", "S", "G2", "M"],
    default="NA"
)
# --------------------------------------------------
# Ordered categorical variable
# --------------------------------------------------
df_wphase["cell_cycle_phase"] = pd.Categorical(
    df_wphase["cell_cycle_phase"],
    categories=["G1", "S", "G2", "M"],
    ordered=True
)

***

In [ ]:
from pathlib import Path
from functools import lru_cache
import numpy as np
import pandas as pd
import tifffile
from skimage import io
from skimage.segmentation import find_boundaries
from skimage.measure import find_contours
from skimage.morphology import binary_dilation, disk
from scipy.spatial import cKDTree

FLUO=['ASH1CLB2','SRL1','SUN4']
MARGIN=12
PCTL=95
CROP_SIZE=None
SQUARE=True
CENTER_ON='mask'
SAVE_MASKS=True
SAVE_SPOTS=False

# Maximum number of cells per cell-cycle phase
# None = use all available cells
MAX_CELLS_PER_PHASE=100
RANDOM_SEED=42
PHASES=['G1','S','G2','M']

MASK_PREFER=['dic']
MASK_EXCLUDE=['dapi','bud']

DRAW_OUTLINE=True
OUTLINE_MODE='outer'
OUTLINE_W=2
DASH_ON=4
DASH_OFF=12

OUTROOT=Path('crops')

DISPLAY_MODE='composite'

CHANNEL_COLORS={
    'ASH1CLB2':'green',
    'SRL1':'magenta',
    'SUN4':'yellow',
    'DAPI':'blue',
    'DIC':'gray',
    'OUTLINE':'red'
}

DEFAULT_COLOR='gray'

CHANNEL_RANGES={
    'ASH1CLB2':(6000,9000),
    'SRL1':(9000,15000),
    'SUN4':(9000,20000),
    'DAPI':(8000,15000),
    'OUTLINE':(0,1)
}

_NAMED={
    'red':(1,0,0),
    'green':(0,1,0),
    'blue':(0,0,1),
    'cyan':(0,1,1),
    'magenta':(1,0,1),
    'yellow':(1,1,0),
    'gray':(1,1,1),
    'orange':(1,0.5,0)
}

def make_lut(color):
    rgb=_NAMED[color] if isinstance(color,str) else tuple(c/255 if c>1 else c for c in color)
    ramp=np.arange(256,dtype=np.uint8)
    return np.stack([(ramp*c).astype(np.uint8) for c in rgb])

@lru_cache(maxsize=4)
def load_fov(basepath_str):
    basepath=Path(basepath_str)
    images={}

    for name in FLUO:
        p=basepath/f'{name}.tif'
        if p.exists():
            images[name]=np.amax(io.imread(p),axis=0)

    images['DAPI']=np.amax(
        io.imread(basepath/'DAPI.tif'),
        axis=0
    )

    dic=io.imread(basepath/'DIC.tif')

    if dic.ndim==3:
        dic=dic[dic.shape[0]//2]

    images['DIC']=dic

    masks_paths=sorted(
        (basepath/'corrected').glob('*.tif')
    )

    masks={
        p.stem:io.imread(p)
        for p in masks_paths
    }

    spot_paths=sorted(
        basepath.glob('*spots.npy')
    )

    have={
        sp.name.split('_')[0]
        for sp in spot_paths
    }

    channels=[
        c for c in FLUO
        if c in images and c in have
    ]+['DAPI','DIC']

    return images,masks,spot_paths,channels

_reported=set()

def cell_mask(masks):
    stems=list(masks)

    ok=[
        s for s in stems
        if not any(
            x in s.lower()
            for x in MASK_EXCLUDE
        )
    ]

    pick=next(
        (
            s for s in ok
            if any(
                x in s.lower()
                for x in MASK_PREFER
            )
        ),
        None
    )

    if pick is None:
        pick=(
            ok[0]
            if ok
            else (stems[0] if stems else None)
        )

    key=tuple(stems)

    if key not in _reported:
        _reported.add(key)
        print(
            f'cell mask: {pick!r} '
            f'(available: {stems})'
        )

    return masks.get(pick)

def cut(a,y0,x0,y1,x1):
    ny,nx=a.shape[-2:]

    ys,ye=max(0,y0),min(ny,y1)
    xs,xe=max(0,x0),min(nx,x1)

    pad=(
        (
            max(0,-y0),
            max(0,y1-ny)
        ),
        (
            max(0,-x0),
            max(0,x1-nx)
        )
    )

    return np.pad(
        a[ys:ye,xs:xe],
        pad,
        mode='constant'
    )

def cell_center(
    masks,
    cell_id,
    fallback_bbox
):
    if CENTER_ON=='mask' and cell_id is not None:
        labels=cell_mask(masks)

        if labels is not None:
            m=labels==cell_id

            if m.any():
                ys,xs=np.nonzero(m)

                return (
                    int(round(ys.mean())),
                    int(round(xs.mean()))
                )

    y0,x0,y1,x1=fallback_bbox

    return (
        (y0+y1)//2,
        (x0+x1)//2
    )

def dashed_outline(
    m,
    mode=OUTLINE_MODE,
    width=OUTLINE_W,
    dash_on=DASH_ON,
    dash_off=DASH_OFF
):
    edge=find_boundaries(
        m,
        mode=mode
    )

    if not edge.any():
        return edge

    if dash_off<=0:
        return (
            binary_dilation(
                edge,
                disk(width-1)
            )
            if width>1
            else edge
        )

    coords=np.argwhere(edge)
    tree=cKDTree(coords)
    out=np.zeros_like(edge)

    period=dash_on+dash_off

    for contour in find_contours(
        m.astype(float),
        0.5
    ):
        step=np.hypot(
            *np.diff(
                contour,
                axis=0
            ).T
        )

        arc=np.concatenate(
            [
                [0.0],
                np.cumsum(step)
            ]
        )

        pts=contour[
            (arc%period)<dash_on
        ]

        if len(pts)==0:
            continue

        _,idx=tree.query(pts)

        sel=coords[idx]

        out[
            sel[:,0],
            sel[:,1]
        ]=True

    if width>1:
        out=binary_dilation(
            out,
            disk(width-1)
        )

    return out

def cell_outline(
    masks,
    cell_id,
    box
):
    by0,bx0,by1,bx1=box

    labels=cell_mask(masks)

    if (
        labels is not None
        and cell_id is not None
    ):
        m=cut(
            (labels==cell_id).astype(
                np.uint8
            ),
            *box
        ).astype(bool)

        if m.any():
            return dashed_outline(m)

    return np.zeros(
        (
            by1-by0,
            bx1-bx0
        ),
        bool
    )

# ============================================================
# SELECT CELLS BY CELL-CYCLE PHASE
# ============================================================

df_phase=df_wphase[
    df_wphase['cell_cycle_phase'].isin(
        PHASES
    )
].copy()

rng=np.random.default_rng(
    RANDOM_SEED
)

selected_parts=[]

for phase in PHASES:
    phase_cells=df_phase[
        df_phase['cell_cycle_phase']==phase
    ].copy()

    n_available=len(
        phase_cells
    )

    if MAX_CELLS_PER_PHASE is None:
        selected=phase_cells

    else:
        n_select=min(
            MAX_CELLS_PER_PHASE,
            n_available
        )

        if n_select>0:
            indices=rng.choice(
                n_available,
                size=n_select,
                replace=False
            )

            selected=phase_cells.iloc[
                indices
            ]

        else:
            selected=phase_cells.iloc[0:0]

    selected_parts.append(
        selected
    )

    print(
        f'{phase}: '
        f'{n_available} available -> '
        f'{len(selected)} selected'
    )

df_selected=pd.concat(
    selected_parts,
    ignore_index=True
)

print(
    f'Total cells selected: '
    f'{len(df_selected)}'
)

# ============================================================
# DETERMINE CROP SIZE
# ============================================================

uids=set(
    df_selected['uid']
)

sub=df_wphase[
    df_wphase['uid'].isin(uids)
]

if CROP_SIZE is not None:
    CROP_H=CROP_W=int(
        CROP_SIZE
    )

else:
    _h=np.percentile(
        sub['bbox-2']-sub['bbox-0'],
        PCTL
    )+2*MARGIN

    _w=np.percentile(
        sub['bbox-3']-sub['bbox-1'],
        PCTL
    )+2*MARGIN

    CROP_H=int(
        np.ceil(_h)
    )

    CROP_W=int(
        np.ceil(_w)
    )

    if SQUARE:
        CROP_H=CROP_W=max(
            CROP_H,
            CROP_W
        )

CROP_H+=CROP_H%2
CROP_W+=CROP_W%2

print(
    f'crop size: '
    f'{CROP_H} x {CROP_W}'
)

# ============================================================
# SAVE CROP
# ============================================================

def save_crop(
    uid,
    basepath,
    outdir,
    y0,
    x0,
    y1,
    x1,
    cell_id=None,
    h=CROP_H,
    w=CROP_W
):
    images,masks,spot_paths,channels=load_fov(
        str(basepath)
    )

    cy,cx=cell_center(
        masks,
        cell_id,
        (y0,x0,y1,x1)
    )

    by0=cy-h//2
    bx0=cx-w//2
    by1=by0+h
    bx1=bx0+w

    box=(
        by0,
        bx0,
        by1,
        bx1
    )

    crops={
        c:cut(
            images[c],
            by0,
            bx0,
            by1,
            bx1
        )
        for c in channels
    }

    channels=list(
        channels
    )

    if DRAW_OUTLINE:
        crops['OUTLINE']=cell_outline(
            masks,
            cell_id,
            box
        )

        channels.append(
            'OUTLINE'
        )

    dtype=np.result_type(
        *[
            crops[c].dtype
            for c in channels
            if crops[c].dtype!=bool
        ]
    )

    stack=np.stack(
        [
            crops[c].astype(dtype)
            for c in channels
        ],
        axis=0
    )

    luts=[
        make_lut(
            CHANNEL_COLORS.get(
                c,
                DEFAULT_COLOR
            )
        )
        for c in channels
    ]

    meta={
        'axes':'CYX',
        'Labels':channels,
        'mode':DISPLAY_MODE,
        'LUTs':luts
    }

    if CHANNEL_RANGES:
        dmax=(
            float(
                np.iinfo(
                    stack.dtype
                ).max
            )
            if np.issubdtype(
                stack.dtype,
                np.integer
            )
            else 1.0
        )

        ranges=[]

        for c,p in zip(
            channels,
            stack
        ):
            lo,hi=CHANNEL_RANGES.get(
                c,
                (
                    float(p.min()),
                    float(p.max())
                )
            )

            if hi<=lo:
                hi=lo+1

            ranges += [
                float(lo),
                min(
                    float(hi),
                    dmax
                )
            ]

        meta['Ranges']=tuple(
            ranges
        )

    tifffile.imwrite(
        outdir/f'cell_{uid}.tif',
        stack,
        imagej=True,
        metadata=meta
    )

    if SAVE_MASKS and masks:
        mstack=np.stack(
            [
                cut(
                    m,
                    by0,
                    bx0,
                    by1,
                    bx1
                )
                for m in masks.values()
            ],
            axis=0
        )

        tifffile.imwrite(
            outdir/f'cell_{uid}_masks.tif',
            mstack.astype(
                np.uint16
            ),
            imagej=True,
            metadata={
                'axes':'CYX',
                'Labels':list(
                    masks
                )
            }
        )

    if SAVE_SPOTS:
        for spot_path in spot_paths:
            yx=np.load(
                spot_path
            )[:,1:3]

            keep=(
                (yx[:,0]>=by0)&
                (yx[:,0]<by1)&
                (yx[:,1]>=bx0)&
                (yx[:,1]<bx1)
            )

            np.save(
                outdir/
                f'cell_{uid}_{spot_path.stem}.npy',
                yx[keep]-
                np.array(
                    [by0,bx0]
                )
            )

    return box

# ============================================================
# RUN CROP EXTRACTION
# ============================================================

written=[]
skipped=[]

for _,r in df_selected.iterrows():
    uid=r['uid']
    condition=r['condition']
    phase=r['cell_cycle_phase']

    rows=df_wphase[
        df_wphase['uid']==uid
    ]

    if rows.empty:
        skipped.append(
            (
                uid,
                'no row in df_wphase'
            )
        )
        continue

    basepath=Path(
        rows['source_folder'].values[0]
    )

    if not basepath.exists():
        skipped.append(
            (
                uid,
                f'missing folder {basepath}'
            )
        )
        continue

    y0=int(
        rows['bbox-0'].iloc[0]
    )

    x0=int(
        rows['bbox-1'].iloc[0]
    )

    y1=int(
        rows['bbox-2'].iloc[0]
    )

    x1=int(
        rows['bbox-3'].iloc[0]
    )

    cell_id=(
        int(
            rows[
                'image_cell_id'
            ].iloc[0]
        )
        if 'image_cell_id' in rows
        else None
    )

    # Folder structure:
    # crops / phase / condition / uid
    outdir=(
        OUTROOT/
        str(phase)/
        str(condition)/
        str(uid)
    )

    outdir.mkdir(
        parents=True,
        exist_ok=True
    )

    try:
        save_crop(
            uid,
            basepath,
            outdir,
            y0,
            x0,
            y1,
            x1,
            cell_id=cell_id
        )

        written.append(uid)

    except Exception as e:
        skipped.append(
            (
                uid,
                repr(e)
            )
        )

# ============================================================
# SUMMARY
# ============================================================

print(
    f'Wrote {len(written)} cells, '
    f'skipped {len(skipped)}'
)

print('\nWritten by phase:')

print(
    df_selected[
        df_selected['uid'].isin(
            written
        )
    ]['cell_cycle_phase']
    .value_counts()
    .sort_index()
)

print('\nWritten by condition:')

print(
    df_selected[
        df_selected['uid'].isin(
            written
        )
    ]['condition']
    .value_counts()
)

if skipped:
    print(
        '\nFirst skipped cells:'
    )

    for s in skipped[:10]:
        print(
            '  ',
            s
        )